# Per-token Natural Indirect Effect (NIE) for steering vectors

Activation-level eval (no GSM8K decoding). For each labelled t-1 position p, computes
`nie_p = log P_steered(pivot|prefix[:p+1]) - log P_base(pivot|prefix[:p+1])`
with the steering perturbation masked to position p only.

Candidate matrix:
- vectors: `caa_val` (Issue #5 bug), `caa_train` (post-fix), `lr_C{0.001..1.0}`.
- modes: `additive_raw`, `additive_normalized`, `projection` (coef=2 doubles, coef=0 patches).

Headline plot (c) tests the GoT prediction: lower-C LR probes have higher cosine to `caa_train` and should show larger NIE.


## 1. Install


In [ ]:
# Run once per runtime. Safe to skip if already installed.
%pip install -q "transformers>=4.44" datasets accelerate matplotlib seaborn scikit-learn torch numpy pandas


## 2. Imports + seed


In [ ]:
import json
import os
import random
import re
import sys
import time
from pathlib import Path

import numpy as np
import torch

try:
    from probe_pipeline.nie import compute_token_nie, cosine
    from probe_pipeline.steering import _get_decoder_layer
    from probe_pipeline.activations import centroid_diff_from_cache
    print("[imports] probe_pipeline available locally")
except Exception:
    print("[imports] probe_pipeline NOT importable; clone the repo (next cell) and re-run")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


## 2b. Colab clone (no-op locally)


In [ ]:
# Colab-friendly clone (no-op locally). After this cell, restart-and-run the
# imports cell above so probe_pipeline.* resolves.
import os, subprocess, sys
REPO_URL = "https://github.com/stvngo/Pivotal-Token-Representation-Learning.git"
REPO_DIR = "Pivotal-Token-Representation-Learning"
if not os.path.isdir(REPO_DIR) and "/content" in os.getcwd():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL])
    sys.path.insert(0, os.path.abspath(REPO_DIR))
elif "probe_pipeline" not in sys.modules:
    here = Path.cwd()
    for parent in [here, *here.parents]:
        if (parent / "probe_pipeline").exists():
            sys.path.insert(0, str(parent))
            break
print("sys.path[0]:", sys.path[0])


## 3. Config


In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
LAYER = 14
MAX_ROWS = int(os.environ.get("MAX_ROWS", "32"))         # cap probe rows for speed
MAX_POSITIONS_PER_ROW = int(os.environ.get("MAX_POSITIONS_PER_ROW", "2"))
RUN_TAG = time.strftime("nie_%Y%m%d_%H%M%S")
RESULTS_DIR = Path("nb_results") / RUN_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"layer={LAYER} max_rows={MAX_ROWS} max_pos/row={MAX_POSITIONS_PER_ROW} -> {RESULTS_DIR}")


## 4. Vector candidate matrix


In [ ]:
# Build the vector candidate matrix:
#   caa_val   <-- old (Issue #5 bug, kept for contrast)
#   caa_train <-- post-fix (Option A)
#   lr_C0.001, lr_C0.01, lr_C0.1, lr_C1.0
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/stvngo/Pivotal-Token-Representation-Learning/main/artifacts/cached3/sklearn/steering_configs"
ART_DIR = Path("artifacts/cached3/sklearn/steering_configs")

def _ensure(name: str) -> Path:
    local = ART_DIR / name if (ART_DIR / name).exists() else Path(name)
    if not local.exists():
        try:
            urllib.request.urlretrieve(f"{REPO_RAW}/{name}", name)
            local = Path(name)
        except Exception as e:
            print(f"  warn: cannot fetch {name}: {e}")
    return local

vectors = {}

p_val = _ensure(f"steering_layer{LAYER}_vector.npy")
if p_val.exists():
    vectors["caa_val"] = np.load(p_val).astype(np.float32).reshape(-1)

p_train = _ensure(f"steering_layer{LAYER}_vector_train.npy")
if p_train.exists():
    vectors["caa_train"] = np.load(p_train).astype(np.float32).reshape(-1)

C_GRID = [0.001, 0.01, 0.1, 1.0]
for C in C_GRID:
    p_w = _ensure(f"steering_layer{LAYER}_probe_weights_C{C}.npy")
    if p_w.exists():
        vectors[f"lr_C{C}"] = np.load(p_w).astype(np.float32).reshape(-1)

print(f"loaded {len(vectors)} candidate vectors:")
for k, v in vectors.items():
    print(f"  {k:>14s} ||v||={np.linalg.norm(v):.3f} dim={v.shape[0]}")

if "caa_train" in vectors:
    base_v = vectors["caa_train"]
    print("\ncosines vs caa_train:")
    for k, v in vectors.items():
        print(f"  {k:>14s} cos={cosine(v, base_v):+.4f}")


## 5. Probe-row dataset (rebuilt from codelion source)


In [ ]:
# Probe rows. compute_token_nie consumes rows of {text, labels[, id]} where
# labels[p]==1 marks a t-1 position (the residual immediately preceding a
# pivotal token). We reuse probe_pipeline.preprocess.create_probe_dataset to
# rebuild from the codelion HF source. This is the same builder
# data_preprocessing.ipynb uses.
from datasets import load_dataset
from probe_pipeline.preprocess import create_probe_dataset

ds = load_dataset("codelion/Qwen3-0.6B-pts-pairs", split="train")
df = ds.to_pandas()
print(f"raw codelion rows: {len(df)}")

# Pull the first MAX_ROWS unique queries.
query_ids = df["dataset_item_id"].unique().tolist()[:MAX_ROWS]
sub = df[df["dataset_item_id"].isin(query_ids)].to_dict(orient="records")
print(f"subset rows={len(sub)} queries={len(query_ids)}")

probe_rows = create_probe_dataset(
    sub, tokenizer=None,         # filled in after model/tokenizer load below
    model=None, device=None,
    add_random_tokens=0, negative_to_positive_ratio=2.0, seed=SEED,
)[:MAX_ROWS]
print(f"probe rows ready for tokenizer: {len(probe_rows)}")


## 6. Load Qwen3-0.6B + tokenize probe rows


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=DTYPE, trust_remote_code=True, low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
print(f"loaded {MODEL_NAME} on {DEVICE}")

# Re-tokenize probe rows with this model's tokenizer. create_probe_dataset
# above ran without a tokenizer for portability, so the labels live on the raw
# `pivot_context + pivot_token` text. Round-trip through the tokenizer here.
probe_rows = create_probe_dataset(
    sub, tokenizer=tokenizer,
    model=None, device=None,
    add_random_tokens=0, negative_to_positive_ratio=2.0, seed=SEED,
)[:MAX_ROWS]
print(f"tokenized probe rows: {len(probe_rows)}")


## 6b. Convention check


In [ ]:
# Convention check (Issue #2 in docs/issues.md).
def _convention_check(_model, _L):
    captured = {}
    def cap(_m, _i, out):
        h = out[0] if isinstance(out, tuple) else out
        captured["pre"] = h.detach()
        return out
    target = _get_decoder_layer(_model, _L)
    handle = target.register_forward_hook(cap)
    try:
        ids = torch.tensor([[1, 2, 3]], device=next(_model.parameters()).device)
        with torch.no_grad():
            outs = _model(input_ids=ids, output_hidden_states=True)
    finally:
        handle.remove()
    return bool(torch.allclose(captured["pre"], outs.hidden_states[_L], atol=1e-4, rtol=1e-3))

CONVENTION_CHECK_PASSED = _convention_check(model, LAYER)
assert CONVENTION_CHECK_PASSED
print(f"[convention] hook == hidden_states[{LAYER}]: OK")


## 7. Run the candidate matrix


In [ ]:
# Candidate matrix: (vector x mode x coef). Each cell costs ~ n_rows * n_pos
# forward passes; smoke runs cap to 32 rows * 2 positions.
modes = [
    ("additive_raw", 0.4),
    ("additive_normalized", 0.2),
    ("projection", 2.0),       # double the existing component along v_hat
    ("projection", 0.0),       # patching: ablate the component
]

results = {}
for vname, vec in vectors.items():
    for mode, coef in modes:
        cell_id = f"{vname}__{mode}__c{coef}"
        print(f"\n=== {cell_id} ===")
        out = compute_token_nie(
            model, tokenizer, probe_rows, LAYER, vec,
            coef=coef, mode=mode, device=DEVICE,
            max_rows=MAX_ROWS, max_positions_per_row=MAX_POSITIONS_PER_ROW,
            label_value=1,
        )
        results[cell_id] = out
        print(f"  n_pos={out['n_positions']} nie_mean={out['nie_mean']:+.4f} "
              f"ci=[{out['bootstrap_ci'][0]:+.3f}, {out['bootstrap_ci'][1]:+.3f}]")


## 8. Plots + save state files


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for cell_id, r in results.items():
    vname, mode, coef_str = cell_id.split("__")
    rows.append({
        "vector": vname, "mode": mode, "coef": float(coef_str.lstrip("c")),
        "n_positions": r["n_positions"],
        "nie_mean": r["nie_mean"], "nie_median": r["nie_median"],
        "ci_lo": r["bootstrap_ci"][0], "ci_hi": r["bootstrap_ci"][1],
        "log_p_base_mean": r["log_p_base_mean"],
        "log_p_steered_mean": r["log_p_steered_mean"],
    })
df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "nie_summary.csv", index=False)
print(df)

# (a) NIE by mode, holding vector = caa_train (or caa_val if missing).
v_keep = "caa_train" if "caa_train" in vectors else "caa_val"
sub = df[df["vector"] == v_keep].copy()
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar([f"{r['mode']}@{r['coef']}" for _, r in sub.iterrows()], sub["nie_mean"])
ax.axhline(0, color="k", lw=0.5)
ax.set_title(f"NIE by mode (vector={v_keep})")
ax.set_ylabel("nie_mean")
fig.tight_layout(); fig.savefig(RESULTS_DIR / "nie_by_mode.png", dpi=120)

# (b) NIE by vector, holding mode = projection alpha=2.
sub = df[(df["mode"] == "projection") & (df["coef"] == 2.0)]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(sub["vector"], sub["nie_mean"])
ax.axhline(0, color="k", lw=0.5)
ax.set_title("NIE by vector @ projection coef=2")
ax.set_ylabel("nie_mean")
fig.tight_layout(); fig.savefig(RESULTS_DIR / "nie_by_vector.png", dpi=120)

# (c) cosine LR<->CAA vs NIE scatter (the headline GoT prediction).
if "caa_train" in vectors:
    base_v = vectors["caa_train"]
    cos_by_v = {k: cosine(v, base_v) for k, v in vectors.items()}
    sub = df[(df["mode"] == "projection") & (df["coef"] == 2.0)]
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter([cos_by_v[v] for v in sub["vector"]], sub["nie_mean"])
    for _, r in sub.iterrows():
        ax.annotate(r["vector"], (cos_by_v[r["vector"]], r["nie_mean"]),
                    fontsize=8, xytext=(3, 3), textcoords="offset points")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xlabel("cos(vector, caa_train)")
    ax.set_ylabel("NIE mean (projection coef=2)")
    ax.set_title("GoT prediction: low-C LR -> high cos -> larger NIE")
    fig.tight_layout(); fig.savefig(RESULTS_DIR / "cosine_vs_nie.png", dpi=120)

# Save state files (per (vector, mode, coef) cell) with provenance.
HOOK_TGT = LAYER - 1 if LAYER >= 1 else -1
for cell_id, r in results.items():
    vname, mode, coef_str = cell_id.split("__")
    coef = float(coef_str.lstrip("c"))
    state = {
        "label": cell_id,
        "metrics": {
            "accuracy": 0.0, "f1": 0.0, "parse_rate": 1.0,
            "correct": 0, "n": r["n_positions"],
            "nie_mean": r["nie_mean"], "nie_median": r["nie_median"],
            "n_positions": r["n_positions"],
        },
        "layer": LAYER, "hook_target_layer_idx": HOOK_TGT,
        "mode": mode, "vector_source": vname,
        "convention_check_passed": CONVENTION_CHECK_PASSED,
        "coef": coef,
        "bootstrap_ci": r["bootstrap_ci"],
        "log_p_base_mean": r["log_p_base_mean"],
        "log_p_steered_mean": r["log_p_steered_mean"],
    }
    (RESULTS_DIR / f"{cell_id.replace('.', '_')}_state.json").write_text(json.dumps(state, indent=2))

print(f"\nsaved -> {RESULTS_DIR}")


## 9. Bundle


In [ ]:
import shutil
zip_path = Path(f"nb_results_{RUN_TAG}.zip")
shutil.make_archive(zip_path.with_suffix(""), "zip", RESULTS_DIR.parent, RESULTS_DIR.name)
print(f"bundle -> {zip_path}")
try:
    from google.colab import files  # noqa: F401
    files.download(str(zip_path))
except Exception:
    pass
